# Assignment 5 — Cramer and LU Decomposition
This notebook compares Cramer's rule, Gauss, and LU decomposition.

## 1. Cramer's Rule — 2×2

In [ ]:
import numpy as np
from linalg_utils.systems import cramer_2x2, cramer_3x3, cramer, resolver_gauss
from linalg_utils.determinants import det_3x3_sarrus
from linalg_utils.lu import decomposicao_lu, det_via_lu, resolver_lu_multiplos_rhs

A = np.array([[3.0, 2.0], [1.0, -1.0]])
b = np.array([7.0, 1.0])
x = cramer_2x2(A, b)
print('A =
', A)
print('b =', b)
print('Solution (Cramer 2x2):', x)
print('NumPy solve:', np.linalg.solve(A, b))

## 2. Cramer's Rule — 3×3

In [ ]:
A = np.array([[2.0, 1.0, -1.0], [3.0, 2.0, 1.0], [1.0, -1.0, 2.0]])
b = np.array([1.0, 12.0, 5.0])
x = cramer_3x3(A, b)
print('Solution (Cramer 3x3):', x)
print('NumPy solve:', np.linalg.solve(A, b))

## 3. Computational Cost — Benchmark

In [ ]:
import timeit

def time_method(fn, A, b, repeats):
    return min(timeit.repeat(lambda: fn(A, b), repeat=3, number=repeats)) / repeats

sizes = [2, 3, 5, 10, 20, 50, 100]
results = []

rng = np.random.default_rng(0)
for n in sizes:
    A = rng.normal(size=(n, n))
    b = rng.normal(size=(n,))
    times = {}
    if n <= 3:
        times['Cramer'] = time_method(cramer, A, b, 10)
    else:
        times['Cramer'] = None
    times['Gauss'] = time_method(lambda A, b: resolver_gauss(A, b)[0], A, b, 10)
    times['NumPy'] = time_method(np.linalg.solve, A, b, 10)
    results.append((n, times))

for n, times in results:
    print(n, times)

# Plot results
import matplotlib.pyplot as plt
labels = ['Cramer', 'Gauss', 'NumPy']
x = np.arange(len(sizes))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 4))
for i, label in enumerate(labels):
    vals = [r[1][label] if r[1][label] is not None else np.nan for r in results]
    ax.bar(x + i * width, vals, width, label=label)

ax.set_xticks(x + width)
ax.set_xticklabels([str(n) for n in sizes])
ax.set_ylabel('Time (s)')
ax.set_xlabel('n')
ax.legend()
plt.tight_layout()
plt.savefig('assets/figures/cramer_benchmark.png', dpi=150)
plt.close()

## 4. LU Decomposition

In [ ]:
A = np.array([[2.0, 1.0, 1.0], [4.0, 3.0, 3.0], [8.0, 7.0, 9.0]])
P, L, U = decomposicao_lu(A)
print('P =
', P)
print('L =
', L)
print('U =
', U)
print('PA =
', P @ A)
print('LU =
', L @ U)
print('det(A) via LU =', det_via_lu(A))
print('det(A) via NumPy =', np.linalg.det(A))

## 5. LU with Multiple Right-Hand Sides

In [ ]:
A = np.array([[2.0, 1.0, 1.0], [4.0, 3.0, 3.0], [8.0, 7.0, 9.0]])
b1 = np.array([4.0, 10.0, 24.0])
b2 = np.array([1.0, 1.0, 1.0])
b3 = np.array([0.0, 2.0, 8.0])
B = np.column_stack([b1, b2, b3])
X = resolver_lu_multiplos_rhs(A, B)
print('Solutions (columns):
', X)
print('Check A @ X:
', A @ X)

## 6. Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, M, title in zip(axes, [A, L, U], ['A', 'L', 'U']):
    im = ax.imshow(M, cmap='viridis')
    ax.set_title(title)
    for (i, j), val in np.ndenumerate(M):
        ax.text(j, i, f'{val:.1f}', ha='center', va='center', color='white')
fig.suptitle('A = LU')
plt.tight_layout()
plt.savefig('assets/figures/lu_heatmap.png', dpi=150)
plt.close()